In [1]:
import sympy as sym
import numpy as np
import random
from scipy.linalg import null_space
from IPython.display import Math, display
import importlib
import sys
import os

# Add the project root directory ('IMU_only') to sys.path
# so Python can locate the 'modules' folder from inside 'scripts/'
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# ab hier ist es so notwendig, um spontane Änderungen in den anderen scripts direkt mit einzuschleusen
import modules.IMU_kinematics_sym
import modules.Speaker_kinematics_sym

importlib.reload(modules.IMU_kinematics_sym)
importlib.reload(modules.Speaker_kinematics_sym)

from modules.IMU_kinematics_sym import *
from modules.Speaker_kinematics_sym import *

sym.init_printing(use_unicode=True)

print(np.__version__)

2.5.2


In this notebook, I will show the observability of my system for both the continuous time and discrete time variants.

In [2]:
# Since in the later application, the amount of faces and available measurements will vary,
# this notebook will take a generalized approach of the state consisting of the IMU and one speaker, while available measurements may vary.

# nominal state combined

x = sym.Matrix.vstack(x_IMU, x_s)
x_dot = sym.Matrix.vstack(x_IMU_dot, x_s_dot)
f_x = sym.Matrix.vstack(f_x_IMU, f_x_s)

# error state combined

delta_x = sym.Matrix.vstack(delta_x_IMU, delta_x_s)
delta_x_n = sym.Matrix.vstack(delta_x_n_IMU, delta_x_s_n)
delta_x_dot = sym.Matrix.vstack(delta_x_IMU_dot, delta_x_s_dot)
f_delta_x = sym.Matrix.vstack(f_delta_x_IMU, f_delta_x_s)

# true state combined

x_true = sym.Matrix.vstack(x_IMU_true, x_s_true)

# measurements combined

# h = sym.Matrix.vstack(g_h, p_h_s, q_h_s)  # without fake measurements
h = sym.Matrix.vstack(g_h, p_h_s, q_h_s, p_h, psi_h)  # with fake measurements
h = sym.simplify(h)

In [3]:
# In order to speed up calculations, I will calculate rank and nullspace of the observability matrix by using randomized substitution.
# This substitution dictionary will make the variables we already know about have a value close to what it will be in the application later

substitution_dict = {
    dt: 0.01,  # approx 100 Hz sampling rate (not fixed in general but for discrete time observability study)
    gravity_sym: gravity_constant,
    IMU_cam_y_translation: IMU_cam_translation_value[1],
    IMU_cam_z_translation: IMU_cam_translation_value[2],
    IMU_cam_w_rotation: IMU_cam_rotation_value[0],
    IMU_cam_x_rotation: IMU_cam_rotation_value[1],
}

## Helper Functions

### Randomized Substitution

In [4]:
def build_substitution_dict(H, F, seed):
    np.random.seed(seed)
    all_symbols = H.free_symbols | F.free_symbols

    # hacky solution, but creating "fake quaternions",
    # only to make the calculations using these values more realistic
    # while keeping their random nature for randomized substitution
    q1_unnormed = [np.random.uniform(-1, 1) for _ in range(4)]
    q2_unnormed = [np.random.uniform(-1, 1) for _ in range(4)]
    q1 = q1_unnormed / np.linalg.norm(q1_unnormed)
    q2 = q2_unnormed / np.linalg.norm(q2_unnormed)
    quat_test_dict = {
        "q_w": q1[0],
        "q_x": q1[1],
        "q_y": q1[2],
        "q_z": q1[3],  # IMU quaternion
        "q_{i,w}": q2[0],
        "q_{i,x}": q2[1],
        "q_{i,y}": q2[2],
        "q_{i,z}": q2[3],  # speaker quaternion
    }

    full_substitution_dict = {}

    for s in all_symbols:
        if s in substitution_dict:
            full_substitution_dict[s] = substitution_dict[s]
        elif s.name in quat_test_dict:
            full_substitution_dict[s] = quat_test_dict[s.name]
        else:
            full_substitution_dict[s] = np.random.uniform(-2, 2)

    return full_substitution_dict

### Observability Matrix Construction

In [5]:
def build_continuous_obs_matrix(n_states, H_delta_x, F_delta_x, seed=10):
    full_substitution_dict = build_substitution_dict(H_delta_x, F_delta_x, seed)

    H_sub = np.array(
        (H_delta_x.xreplace(full_substitution_dict)).tolist(), dtype=np.float64
    )
    F_sub = np.array(
        (F_delta_x.xreplace(full_substitution_dict)).tolist(), dtype=np.float64
    )

    obs_matrix = H_sub
    HF_k = H_sub
    for k in range(n_states):
        HF_k = HF_k @ F_sub
        obs_matrix = np.vstack((obs_matrix, HF_k))

    return obs_matrix


def build_discrete_obs_matrix(n_states, H_delta_x, F_delta_x, base_seed=10):
    """
    Builds the discrete-time ESEKF observability matrix cleanly by first
    generating the trajectory matrices, then accumulating the state transition chain.
    """
    H_matrices = []
    F_matrices = []

    # Step 1: Evaluate H and F along the trajectory step-by-step
    for k in range(n_states):
        sub_dict = build_substitution_dict(H_delta_x, F_delta_x, seed=base_seed + k)

        # Evaluate H at time step k
        H_k = np.array((H_delta_x.xreplace(sub_dict)).tolist(), dtype=np.float64)
        H_matrices.append(H_k)

        # Evaluate F at time step k (needed for steps 0 to n-2)
        if k < n_states - 1:
            F_k = np.array((F_delta_x.xreplace(sub_dict)).tolist(), dtype=np.float64)
            F_matrices.append(F_k)

    # Step 2: Build the observability matrix rows intuitively
    obs_rows = [H_matrices[0]]  # Row 0: H(k_0)

    F_prod = None
    for k in range(1, n_states):
        # Accumulate the transition chain: F(k-1) * ... * F(0)
        # F_matrices[k-1] corresponds to F(k_0 + k - 1)
        if k == 1:
            F_prod = F_matrices[0]
        else:
            F_prod = F_matrices[k - 1] @ F_prod

        # Row k: H(k_0 + k) * [F chain product]
        obs_row = H_matrices[k] @ F_prod
        obs_rows.append(obs_row)

    return np.vstack(obs_rows)

### Nullspace Visualization

In [6]:
def visualize_nullspace(nullspace, threshold):
    if nullspace.shape[1] <= 1:
        cleaned_nullspace = nullspace
    else:  # RREF
        cleaned_nullspace, _ = sym.Matrix(nullspace.T).rref()
        cleaned_nullspace = np.array(cleaned_nullspace, dtype=float).T

    cleaned_nullspace = np.where(
        np.abs(cleaned_nullspace) < threshold, 0, np.round(cleaned_nullspace, 4)
    )

    # for symbolic visualization
    state_col_mat = np.array(delta_x).reshape(-1, 1)
    nullspace_sign = np.sign(cleaned_nullspace).astype(int)
    symbolic_nullspace = state_col_mat * nullspace_sign

    latex_nullspace = sym.latex(sym.Matrix(nullspace))
    latex_cleaned = sym.latex(sym.Matrix(cleaned_nullspace))
    latex_symbolic = sym.latex(sym.Matrix(symbolic_nullspace))

    display(
        Math(
            f"""
    \\begin{{matrix}}
    \\text{{Column Vector{"" if (nullspace.shape[1] <= 1) else "s"} spanning Nullspace}} & \\quad & \\text{{RREF of Nullspace}} & \\quad & \\text{{Symbolic State Nullspace}} \\\\
    \\\\ {latex_nullspace} & & {latex_cleaned} & & {latex_symbolic}
    \\end{{matrix}}
    """
        )
    )

## Continuous Time Observability

In [7]:
# h = sym.Matrix.vstack(g_h, q_h_s, p_h_s, p_h, psi_h)
h = sym.Matrix.vstack(g_h, q_h_s, p_h_s, p_h, psi_h)

n_states = delta_x.shape[0]

# jacobians
H_delta_x = h.jacobian(x) * x_true.jacobian(delta_x)
F_delta_x = delta_x_dot.jacobian(delta_x)
# Substitute zero error (see markdown explanation)
H_delta_x = H_delta_x.subs({var: 0 for var in delta_x})
F_delta_x = F_delta_x.subs({var: 0 for var in delta_x})

obs_matrix = build_continuous_obs_matrix(n_states, H_delta_x, F_delta_x)


U, S, Vh = np.linalg.svd(obs_matrix, full_matrices=False)
threshold = (
    max(obs_matrix.shape) * np.finfo(obs_matrix.dtype).eps * S[0]
)  # threshold = 4.58930124362761e-12

# rank
rank = int(np.sum(S > threshold))
print(
    f"rank: {rank} / {n_states} | {n_states - rank} unobservable state{"" if (n_states - rank == 1) else "s"}"
)

# nullspace
if rank < n_states:
    nullspace = Vh[rank:].T

    visualize_nullspace(nullspace, threshold)

rank: 24 / 24 | 0 unobservable states


## Discrete Time Observability

In [8]:
n_states = delta_x.shape[0]

# jacobians
H_delta_x = h.jacobian(x) * x_true.jacobian(delta_x)
F_delta_x = delta_x_dot.jacobian(delta_x)
# Substitute zero error
H_delta_x = H_delta_x.subs({var: 0 for var in delta_x})
F_delta_x = F_delta_x.subs({var: 0 for var in delta_x})

# Build discrete observability matrix along a trajectory
obs_matrix = build_discrete_obs_matrix(n_states, H_delta_x, F_delta_x, base_seed=42)

# SVD and rank analysis
U, S, Vh = np.linalg.svd(obs_matrix, full_matrices=False)
threshold = max(obs_matrix.shape) * np.finfo(obs_matrix.dtype).eps * S[0]

rank = int(np.sum(S > threshold))
print(
    f"rank: {rank} / {n_states} | {n_states - rank} unobservable state{'' if (n_states - rank == 1) else 's'}"
)

if rank < n_states:
    nullspace = Vh[rank:].T
    visualize_nullspace(nullspace, threshold)

rank: 24 / 24 | 0 unobservable states


## Testing

### Does the randomized substitution approach hold up for several rounds

In [9]:
%%script true
timer = Timer()

n_states = delta_x.shape[0]
H_delta_x, F_delta_x = build_jacobians()

for i in range(10,30):
    obs_matrix = build_continuous_obs_matrix(n_states, H_delta_x, F_delta_x, i)

    U, S, Vh = np.linalg.svd(obs_matrix, full_matrices=False)
    threshold = max(obs_matrix.shape) * np.finfo(obs_matrix.dtype).eps * S[0] # threshold = 4.58930124362761e-12

    # rank
    rank = int(np.sum(S > threshold))
    print(f"final rank: {rank} / {n_states} | {n_states - rank} unobservable state{"" if (n_states - rank == 1) else "s"} | {timer.check()}")

    # nullspace
    if rank < n_states:
        nullspace = Vh[rank:].T

        visualize_nullspace(nullspace, threshold)